# 07 - Observability

Every notebook in this pipeline (01-06) logs through `RunLogger` (`src/observability/run_logger.py`) - each run writes a structured JSON record to `datasets/reports/runs/` with per-stage timing, status, and metrics. This notebook reads those back and renders the pipeline's actual execution history, plus the architecture/dataflow diagram (`src/observability/diagram.py`).

In [1]:
import sys
sys.path.insert(0, r"d:\project-raw-data\sphoorthq-geoverse")

import pandas as pd

from src.observability.run_logger import load_recent_runs
from src.observability.diagram import ARCHITECTURE_DIAGRAM

## Architecture / dataflow diagram

Rendered from `src/observability/diagram.py` (Mermaid) - Jupyter renders Mermaid natively when the `mermaid`/`ipymermaid` extension is available; the raw source always prints below regardless, and the same string is embedded in `docs/architecture.md`.

In [2]:
print(ARCHITECTURE_DIAGRAM)

flowchart TB
    subgraph SOURCES["Cloud Sources"]
        S3["AWS S3"]
        ADLS["Azure ADLS Gen2"]
        GCS["Google Cloud Storage"]
        HTTP["Public HTTPS bucket"]
    end

    subgraph INGEST["src/ingestion"]
        CONN["IngestionConnector\n(s3 / adls / gcs / http / local)"]
    end

    RAW[("datasets/raw/\nlocal, analysis-ready")]

    subgraph PROCESS["notebooks/02_process\n+ src/processing"]
        CAL["SAR calibration\n+ speckle filter"]
        COREG["Co-registration\n(common grid/CRS)"]
    end

    subgraph FEAT["notebooks/03_feature_engineering"]
        TEX["Texture / polarimetric\nfeatures"]
        VEC["Per-pixel / superpixel\nfeature vectors"]
    end

    subgraph CLASSICAL["Classical ML"]
        UNET["U-Net segmentation"]
    end

    subgraph QUANTUM["Quantum ML - src/qml"]
        IBM["IBM Quantum Runtime\n(QiskitRuntimeService)"]
        BRAKET["AWS Braket\n(AwsDevice / LocalSimulator)"]
        QKERNEL["Quantum kernel SVM"]
    end

    HYBRID["Hybri

## Recent pipeline runs

One row per stage, across every notebook run so far. Run notebooks 01-06 first if this is empty.

In [3]:
runs = load_recent_runs(limit=50)
print(f"{len(runs)} run records found in datasets/reports/runs/")

rows = []
for run in runs:
    for stage in run["stages"]:
        rows.append({
            "run_name": run["run_name"],
            "stage": stage["name"],
            "status": stage["status"],
            "duration_s": stage["duration_s"],
            **stage.get("metrics", {}),
        })

df = pd.DataFrame(rows)
df

6 run records found in datasets/reports/runs/


,run_name,stage,status,duration_s,n_train,n_test,iou,f1,precision,recall,...,n_pixels,n_estimators,feature_channels,n_valid_pixels,water_fraction,shape,vv_std_reduction,remote_object_count,remote_size_mb,sources_found
0,06_hybrid_ensemble_evaluation,rebuild_sample,ok,0.052,4.0,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,06_hybrid_ensemble_evaluation,load_classical_model,ok,0.227,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,06_hybrid_ensemble_evaluation,train_quantum_kernel,ok,519.961,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,06_hybrid_ensemble_evaluation,hybrid_vote,ok,0.000,NaN,NaN,0.5000,0.6667,0.5000,1.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,06_hybrid_ensemble_evaluation,robustness_speckle,ok,1.624,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,05_qml_ibm_braket,build_small_labeled_sample,ok,0.062,4.0,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,05_qml_ibm_braket,check_ibm_quantum_connectivity,ok,3.143,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,05_qml_ibm_braket,check_braket_connectivity,ok,0.000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,05_qml_ibm_braket,train_quantum_kernel_svm_ibm,ok,318.602,NaN,NaN,0.0000,0.0000,0.0000,0.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,05_qml_ibm_braket,train_quantum_kernel_svm_braket,ok,2.081,NaN,NaN,0.5000,0.6667,0.5000,1.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
if not df.empty:
    print("Total stage time per notebook run:")
    print(df.groupby("run_name")["duration_s"].sum().round(2))
    print()
    print("Any failed stages:")
    failed = df[df["status"] == "failed"]
    print(failed if not failed.empty else "none")

Total stage time per notebook run:
run_name
01_ingest                          1.43
02_process                         0.11
03_feature_engineering             0.06
04_classical_ml                  856.35
05_qml_ibm_braket                323.89
06_hybrid_ensemble_evaluation    521.86
Name: duration_s, dtype: float64

Any failed stages:
none
